# [M1] NÓI CHUYỆN VỚI DỮ LIỆU BẰNG PYTHON
## Buổi S03: Làm Sạch Dữ Liệu: Missing Values, Data Types & Duplicates

---

### Metadata Khóa Học
* **Module:** M1 — NÓI CHUYỆN VỚI DỮ LIỆU BẰNG PYTHON
* **Buổi:** S03
* **Thời lượng dự kiến:** 120 phút
* **Sản phẩm đầu ra (Session Output):** Notebook làm sạch dữ liệu toàn diện loại bỏ NaNs và Duplicates
* **Mức độ liên quan VOAI / Olympic AI:** Bảo đảm tính toàn vẹn dữ liệu, triệt tiêu lỗi sập code do thiếu dữ liệu khi thi đấu.

#### Mục tiêu học tập (Learning Objectives)
- Phát hiện và định lượng tỷ lệ dữ liệu khuyết (Missing Values / NaNs) trên từng cột.
- Hiểu và áp dụng đúng đắn các chiến lược điền khuyết (Imputation): Mean, Median, Mode, hoặc Constant.
- Nhận diện và sửa lỗi sai lệch kiểu dữ liệu (Data Types Mismatch) bằng `.astype()` và `pd.to_datetime()`.
- Phát hiện, phân tích và loại bỏ các dòng dữ liệu trùng lặp (Duplicate Records).

#### Điều kiện tiên quyết (Prerequisites)
- Hoàn thành Buổi M1_S02 (Pandas Core & Indexing).
- Hiểu khái niệm giá trị rỗng (None / NaN) trong lập trình.

---


## 1. WHY? — TẠI SAO PHẢI HỌC CHỦ ĐỀ NÀY?


Trong phòng thi Olympic AI, ban tổ chức gần như không bao giờ cung cấp một bộ dữ liệu hoàn hảo. Thậm chí, họ cố tình đưa vào các giá trị bị khuyết (`NaN`), các cột số bị lưu nhầm dưới dạng chuỗi văn bản (`object`), hoặc các bản ghi bị sao chép trùng lặp.

Hầu hết các thuật toán Machine Learning (như Linear Regression, SVM, mạng nơ-ron PyTorch) sẽ **lập tức báo lỗi Crash** nếu dữ liệu đầu vào chứa dù chỉ một giá trị `NaN`. Nếu bạn xử lý ẩu bằng cách xóa sạch mọi dòng có NaN, bạn có thể làm mất đi 50% dữ liệu quý giá. Làm chủ kỹ thuật làm sạch dữ liệu là rào chắn đầu tiên bảo vệ mô hình của bạn.



## 2. CONCEPT — NỀN TẢNG LÝ THUYẾT & TRỰC GIÁC


### 1. Cơ chế Missing Data trong Pandas
* Giá trị khuyết thường được biểu diễn bằng `np.nan` (Not a Number - kiểu float) hoặc `None`.
* Phát hiện: `df.isna()` hoặc `df.isnull()`.
* Đếm số lượng khuyết: `df.isna().sum()`.
* Tỷ lệ phần trăm khuyết: `df.isna().mean() * 100`.

### 2. Các chiến lược xử lý dữ liệu khuyết (Imputation Strategies)
1. **Xóa bỏ (Dropping):** `df.dropna()`.
   - *Khi nào nên dùng:* Khi tỷ lệ khuyết quá nhỏ (< 1-2%) và tập dữ liệu đủ lớn, HOẶC khi một cột bị khuyết trên 70% không còn giá trị thông tin.
2. **Điền giá trị trung tâm (Mean / Median Imputation):**
   - Dùng `Mean` khi dữ liệu có phân phối chuẩn (chuông đối xứng).
   - Dùng `Median` khi dữ liệu có phân phối lệch (Skewed) hoặc có ngoại lai (Outliers).
3. **Điền giá trị xuất hiện nhiều nhất (Mode Imputation):**
   - Bắt buộc dùng cho biến phân loại (Categorical data, ví dụ: Thành phố, Giới tính).
4. **Cảnh báo Data Leakage:** Khi điền khuyết, giá trị Mean/Median phải được tính **chỉ trên tập Train**, sau đó mới áp dụng giá trị đó để điền cho tập Test/Validation!

### 3. Xử lý kiểu dữ liệu (Type Casting) & Trùng lặp (Duplicates)
* Đổi kiểu dữ liệu: `df['col'] = df['col'].astype(int)` hoặc `pd.to_numeric(df['col'], errors='coerce')`.
* Kiểm tra trùng lặp: `df.duplicated().sum()`.
* Xóa trùng lặp: `df.drop_duplicates(keep='first', inplace=True)`.



## 3. VISUAL INTUITION — TRỰC QUAN HÓA KIẾN THỨC


```
Quy trình làm sạch dữ liệu chuẩn (Cleaning Pipeline):
[Raw Data] ──> [Phát hiện NaNs] ──> [Imputation / Drop] 
                 │
                 ├──> [Ép đúng Dtype (str -> float)] 
                 │
                 └──> [Xóa Duplicates] ──> [Clean Dataset sẵn sàng cho ML]
```



## 4. CODE IMPLEMENTATION & 5. WALKTHROUGH

*Toàn bộ code dưới đây được thiết kế thực tế, có thể chạy độc lập không phụ thuộc tải dữ liệu bên ngoài.*

In [ ]:
import pandas as pd
import numpy as np

# 1. Tao tap du lieu "ban" mo phong thuc te
raw_data = {
    'Candidate_ID': ['C01', 'C02', 'C03', 'C04', 'C05', 'C06', 'C03'], # C03 bi trung lap
    'Age': [19, np.nan, 21, 20, np.nan, 22, 21],
    'Salary_Expectation': ['$1,200', '$1,500', '$1,100', 'MISSING', '$2,000', '$1,800', '$1,100'], # Co ky tu '$' va 'MISSING'
    'City': ['Hanoi', 'HCMC', np.nan, 'Danang', 'Hanoi', 'HCMC', np.nan],
    'Score': [85.0, 90.0, 78.0, np.nan, 88.0, 92.0, 78.0]
}

df_dirty = pd.DataFrame(raw_data)
print("===== DU LIEU THO BAN DAU =====")
print(df_dirty)

print("\n===== THONG KE SO LUONG GIA TRI KHUYET (NANS) =====")
print(df_dirty.isna().sum())

### 🔍 Code Walkthrough (Khối 1)
Khối code tạo dựng một DataFrame chứa đầy đủ các "bệnh" điển hình của dữ liệu thực tế:
- Cột `Age` và `Score` bị khuyết `NaN`.
- Cột `City` bị khuyết dữ liệu dạng chuỗi (Categorical).
- Cột `Salary_Expectation` chứa ký tự đặc biệt `$` và chữ `'MISSING'`, khiến Pandas nhận diện nhầm thành kiểu `object` (chuỗi).
- Bản ghi `C03` bị lặp lại 2 lần.


In [ ]:
# 2. Pipeline lam sach du lieu toan dien
df_clean = df_dirty.copy()

# Buoc 1: Loai bo ban ghi trung lap (Duplicates)
before_dedup = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['Candidate_ID'], keep='first')
print(f"Da loai bo {before_dedup - len(df_clean)} dong trung lap.")

# Buoc 2: Xu ly cot Salary_Expectation: Bo '$', thay 'MISSING' thanh NaN roi ep sang float
df_clean['Salary_Expectation'] = (
    df_clean['Salary_Expectation']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .replace('MISSING', np.nan)
)
df_clean['Salary_Expectation'] = pd.to_numeric(df_clean['Salary_Expectation'])

# Buoc 3: Dien khuyet (Imputation)
# Dien Age bang Median
median_age = df_clean['Age'].median()
df_clean['Age'] = df_clean['Age'].fillna(median_age)

# Dien City bang Mode (Gia tri xuat hien nhieu nhat)
mode_city = df_clean['City'].mode()[0]
df_clean['City'] = df_clean['City'].fillna(mode_city)

# Dien Score bang Mean
mean_score = df_clean['Score'].mean()
df_clean['Score'] = df_clean['Score'].fillna(mean_score)

print("\n===== BANG DU LIEU SAU KHI LAM SACH HOAN TOAN =====")
print(df_clean)
print("\nKiem tra lai so luong NaNs con lai:", df_clean.isna().sum().sum())

### 🔍 Code Walkthrough (Khối 2)
Khối code thực thi toàn bộ pipeline làm sạch một cách tuần tự:
- Khử trùng lặp dựa trên khóa chính `Candidate_ID`.
- Sử dụng chuỗi xử lý chuỗi (`.str.replace`) và `pd.to_numeric` để chuyển hóa một cột rác về đúng dạng số thực `float64`.
- Sử dụng chính xác chiến lược điền khuyết: `Median` cho tuổi, `Mode` cho địa phương, `Mean` cho điểm thi.
- Kiểm tra tính toàn vẹn ở bước cuối cùng (`isna().sum().sum() == 0`).


## 6. PRACTICE — BÀI TẬP THỰC HÀNH


### Bài 1 (Easy): Báo cáo tỷ lệ khuyết
Viết một hàm `missing_report(df)` nhận vào một DataFrame và trả về một bảng thống kê gồm 2 cột: `Missing_Count` (số lượng khuyết) và `Missing_Ratio_%` (tỷ lệ phần trăm khuyết sắp xếp giảm dần).

### Bài 2 (Easy): Xử lý khuyết bằng Forward Fill
Tìm hiểu và thực thi phương thức `df.fillna(method='ffill')` hoặc `df.ffill()`. Giải thích trong dạng dữ liệu nào (ví dụ: chuỗi thời gian - Time Series, giá chứng khoán) thì kỹ thuật này lại cực kỳ hữu ích.

### Bài 3 (Medium): Phát hiện giá trị rác ngầm định
Trong nhiều cuộc thi, người ra đề mã hóa giá trị khuyết thành các số dị thường như `-999`, `9999`, hoặc chuỗi `'?'`, `'N/A'`. Hãy viết hàm tự động tìm và thay thế tất cả các giá trị `-999` trong các cột số thành `np.nan`.

### Bài 4 (Medium): Đổi kiểu ngày tháng (Datetime Parsing)
Cho cột `Date = ['2026-01-15', '2026/02/20', '15-03-2026']`. Sử dụng `pd.to_datetime()` để chuẩn hóa toàn bộ về kiểu `datetime64[ns]`, sau đó trích xuất ra 2 cột mới: `Year` và `Month`.

### Bài 5 (Challenge): Viết Class Transformer làm sạch dữ liệu
Tạo một lớp Python `DataCleaner` có hai phương thức chuẩn phong cách Scikit-Learn:
- `fit(df)`: Học và lưu trữ lại Median của các cột số và Mode của các cột danh mục từ tập dữ liệu.
- `transform(df)`: Dùng các giá trị đã học ở bước `fit` để điền khuyết cho một DataFrame mới mà không gây ra Data Leakage.



## 7. VOAI CONNECTION — CHIẾN LƯỢC PHÒNG THI OLYMPIC


Một trong những cái bẫy tinh vi nhất trong kỳ thi Olympic AI là **Data Leakage trong quá trình Imputation**:
Nhiều thí sinh gộp cả file `train.csv` và `test.csv` lại rồi mới tính Mean để điền khuyết. Điều này đồng nghĩa với việc bạn đã cho mô hình "nhìn trộm" phân phối của tập kiểm tra trước khi thi đấu, dẫn đến điểm kiểm thử cục bộ cực cao nhưng điểm nộp bài thực tế lại thảm hại. Quy tắc vàng: **Chỉ fit trên Train, áp dụng sang Test!**



## 8. DEBUGGING CHALLENGE — BẮT LỖI THỰC CHIẾN


**Tình huống thực tế:** Thí sinh cố gắng ép kiểu một cột dữ liệu sang số nguyên nhưng bị văng lỗi:

```python
# CODE CÓ LỖI CASTING KHI CHỨA NAN:
s = pd.Series([1, 2, np.nan, 4])
# s_int = s.astype(int) # Loi: ValueError: Cannot convert non-finite values (NA or inf) to integer
```

**Nhiệm vụ của bạn:**
1. Giải thích tại sao kiểu dữ liệu nguyên thủy `int` trong NumPy cũ không thể chứa giá trị `NaN`.
2. Sửa lỗi bằng cách sử dụng kiểu số nguyên mở rộng của Pandas (`Int64` với chữ I viết hoa) hoặc điền khuyết trước khi ép kiểu.



## 9. KNOWLEDGE CHECK — KIỂM TRA ĐỘ HIỂU BẢN CHẤT


1. Khi một cột dữ liệu bị khuyết nhiều giá trị và có sự xuất hiện của các số cực đoan (outliers), ta nên điền khuyết bằng đại lượng nào?
   - A. Giá trị 0
   - B. Giá trị trung bình cộng (Mean)
   - C. Giá trị trung vị (Median)
   - D. Luôn luôn xóa toàn bộ cột đó

2. Lệnh `df.drop_duplicates(keep='last')` có ý nghĩa gì?
   - A. Xóa tất cả các dòng bị trùng lặp, không giữ lại dòng nào.
   - B. Giữ lại bản ghi xuất hiện cuối cùng trong các bản ghi bị trùng, xóa các bản ghi phía trước.
   - C. Chỉ xóa dòng cuối cùng của bảng dữ liệu.
   - D. Báo lỗi nếu có bản ghi trùng lặp.

3. Phương thức nào của Pandas giúp chuyển đổi một chuỗi văn bản không chuẩn hóa sang số thực và biến các ký tự lỗi thành NaN?
   - A. `pd.to_numeric(s, errors='coerce')`
   - B. `s.to_float()`
   - C. `float(s)`
   - D. `pd.cast_number(s)`

4. Vì sao tính Mean trên toàn bộ tập dữ liệu (cả Train và Test) rồi điền khuyết lại bị coi là phạm luật kỹ thuật Data Leakage?
   - A. Vì làm tăng thời gian chạy của CPU.
   - B. Vì vi phạm nguyên tắc tập Test phải hoàn toàn vô hình đối với mô hình trong quá trình học.
   - C. Vì khiến cho Pandas bị tràn bộ nhớ.
   - D. Vì kết quả luôn luôn bằng 0.

5. Cột có kiểu dữ liệu là gì nếu nó chứa cả số thực lẫn chuỗi văn bản trong Pandas?
   - A. `float64`
   - B. `string`
   - C. `object`
   - D. `mixed`



## 10. SESSION SUMMARY — TỔNG KẾT BUỔI HỌC


* **What I learned:**
  - Kỹ thuật phát hiện và đo lường tỷ lệ Missing Data.
  - Các chiến lược điền khuyết (Mean, Median, Mode) và phòng tránh rò rỉ dữ liệu.
  - Xử lý xung đột kiểu dữ liệu và khử trùng lặp dữ liệu bảng.
* **What I can now do:**
  - Tự xây dựng một Data Cleaning Pipeline hoàn chỉnh cho dữ liệu thô.
  - Đảm bảo dữ liệu không còn giá trị khuyết bất hợp lệ trước khi đưa vào mô hình.
* **What I still need to practice:**
  - Nâng cao kỹ năng phân tích nhóm dữ liệu (GroupBy) và biểu diễn bằng biểu đồ trực quan ở Buổi 4.



## 11. NEXT SESSION — BƯỚC ĐI TIẾP THEO


Trong **Buổi M1_S04**, chúng ta sẽ học cách "đọc vị" những câu chuyện ẩn sâu trong bảng dữ liệu: **Thống Kê Mô Tả, Phân Tích Nhóm (GroupBy) & Trực Quan Hóa Dữ Liệu (Data Visualization)** với Matplotlib và Seaborn!



---

## 🎯 Syllabus Traceability (Đối chiếu Syllabus)

Buổi học này đóng góp trực tiếp vào mục tiêu của **M1**:
- Làm sạch dữ liệu & phát hiện NaNs ✓
- Xử lý missing values bằng Mean, Median, Mode ✓
- Ép kiểu dữ liệu (Data Types) & loại bỏ Duplicates ✓

* **Session Deliverable:** Mã nguồn Data Cleaning Pipeline chuẩn mực không rò rỉ dữ liệu
* **Đảm bảo chuẩn Syllabus Lock:** Nội dung nằm đúng chuẩn 100%, không lan man, giải quyết trực tiếp năng lực cốt lõi.
